In [1]:
from pathlib import Path

pdf_folder = Path("../reports/raw_pdfs")

pdf_files = sorted(pdf_folder.glob("*.pdf"))

print(f"Number of PDF reports found: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Number of PDF reports found: 20
001_barclays_plc_annual_report_2025.pdf
002_hsbc_holdings_plc_annual_report_2025.pdf
003_lloyds_banking_group_plc_annual_report_2025.pdf
004_natwest_group_plc_annual_report_2025.pdf
005_standard_chartered_plc_annual_report_2025.pdf
006_aviva_plc_annual_report_2025.pdf
007_legal_and_general_group_plc_annual_report_2025.pdf
008_m_and_g_plc_annual_report_2025.pdf
009_admiral_group_plc_annual_report_2025.pdf
010_aj_bell_plc_annual_report_2025.pdf
011_ig_group_holdings_plc_annual_report_2025.pdf
012_cmc_markets_plc_annual_report_2025.pdf
013_integrafin_holdings_plc_annual_report_2025.pdf
014_wise_plc_annual_report_2025.pdf
015_cab_payments_holdings_plc_annual_report_2025.pdf
016_funding_circle_holdings_plc_annual_report_2025.pdf
017_boku_inc_annual_report_2025.pdf
018_paypoint_plc_annual_report_2025.pdf
019_london_stock_exchange_group_plc_annual_report_2025.pdf
020_experian_plc_annual_report_2025.pdf


In [2]:
%pip install pymupdf

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ------ --------------------------------- 3.1/19.8 MB 21.3 MB/s eta 0:00:01
   ------------------ --------------------- 9.2/19.8 MB 25.0 MB/s eta 0:00:01
   ----------------------------- ---------- 14.7/19.8 MB 25.5 MB/s eta 0:00:01
   ---------------------------------------  19.7/19.8 MB 25.7 MB/s eta 0:00:01
   ---------------------------------------  19.7/19.8 MB 25.7 MB/s eta 0:00:01
   ---------------------------------------- 19.8/19.8 MB 17.2 MB/s  0:00:01


In [3]:
import fitz

# Select the first report: Barclays
sample_pdf = pdf_files[0]

# Open the PDF and extract text from the first page
with fitz.open(sample_pdf) as document:
    print(f"File tested: {sample_pdf.name}")
    print(f"Number of pages: {len(document)}")
    
    first_page_text = document[0].get_text("text")

print("\nFirst-page text preview:\n")
print(first_page_text[:1500])

File tested: 001_barclays_plc_annual_report_2025.pdf
Number of pages: 494

First-page text preview:

Barclays PLC  |  Annual Report 2025



In [4]:
# Check the first 20 pages and identify the page
# containing the largest amount of extractable text

with fitz.open(sample_pdf) as document:
    
    page_lengths = []

    for page_index in range(min(20, len(document))):
        page_text = document[page_index].get_text("text", sort=True)
        page_lengths.append((page_index + 1, len(page_text)))

    best_page_number, character_count = max(
        page_lengths,
        key=lambda item: item[1]
    )

    best_page_text = document[
        best_page_number - 1
    ].get_text("text", sort=True)

print(f"Text-heavy page selected: {best_page_number}")
print(f"Number of characters: {character_count}")

print("\nText preview:\n")
print(best_page_text[:2000])

Text-heavy page selected: 20
Number of characters: 10087

Text preview:

   Strategic      Shareholder     Climate and                               Risk         Financial       Financial                                                                                                Barclays PLC
   report         information       sustainability report     Governance     review      review       statements                                                                                          Annual Report 2025                                                                                      19
2025 divisional review (continued)


                                                                                    We’ve also reduced client onboarding times by     Our extensive client relationships and experience
  Supporting clients                                                                     around 50% since the start of 2024 and, in          of UK payments will benefit from 

In [5]:
import pandas as pd

# Define output locations

output_folder = Path("../data/extracted_text")
log_file = Path("../data/processed/text_extraction_log.csv")

# Create the folders if they do not already exist

output_folder.mkdir(parents=True, exist_ok=True)
log_file.parent.mkdir(parents=True, exist_ok=True)

# Store extraction information

extraction_results = []

print("Beginning text extraction...\n")

# Process every annual report

for report_number, pdf_path in enumerate(pdf_files, start=1):

    print(
        f"[{report_number}/{len(pdf_files)}] "
        f"Processing: {pdf_path.name}"
    )

    try:

        page_texts = []
        pages_with_text = 0

        with fitz.open(pdf_path) as document:

            total_pages = len(document)

            for page_number, page in enumerate(
                document,
                start=1
            ):

                # Extract text using approximate reading order

                text = page.get_text(
                    "text",
                    sort=True
                )

                if text.strip():
                    pages_with_text += 1

                # Preserve the original PDF page number

                page_texts.append(
                    f"\n\n"
                    f"===== PAGE {page_number} ====="
                    f"\n\n{text}"
                )

        # Combine all pages

        complete_text = "".join(page_texts)

        # Create output filename

        output_path = (
            output_folder
            / f"{pdf_path.stem}.txt"
        )

        # Save extracted text

        output_path.write_text(
            complete_text,
            encoding="utf-8"
        )

        # Record successful extraction

        extraction_results.append({

            "file_name": pdf_path.name,

            "page_count": total_pages,

            "pages_with_text":
                pages_with_text,

            "empty_or_image_only_pages":
                total_pages
                - pages_with_text,

            "character_count":
                len(complete_text),

            "output_file":
                output_path.name,

            "extraction_status":
                "Success",

            "error_message":
                ""

        })

        print(
            f"Completed: "
            f"{total_pages} pages extracted\n"
        )

    except Exception as error:

        extraction_results.append({

            "file_name":
                pdf_path.name,

            "page_count":
                None,

            "pages_with_text":
                None,

            "empty_or_image_only_pages":
                None,

            "character_count":
                None,

            "output_file":
                "",

            "extraction_status":
                "Failed",

            "error_message":
                str(error)

        })

        print(
            f"Extraction failed: "
            f"{error}\n"
        )

# Convert the extraction results into a table

extraction_log = pd.DataFrame(
    extraction_results
)

# Save the extraction log

extraction_log.to_csv(
    log_file,
    index=False
)

print(
    "All reports have been processed."
)

print(
    f"\nExtraction log saved to:\n"
    f"{log_file}"
)

# Display summary

extraction_log

Beginning text extraction...

[1/20] Processing: 001_barclays_plc_annual_report_2025.pdf
Completed: 494 pages extracted

[2/20] Processing: 002_hsbc_holdings_plc_annual_report_2025.pdf
Completed: 372 pages extracted

[3/20] Processing: 003_lloyds_banking_group_plc_annual_report_2025.pdf
Completed: 327 pages extracted

[4/20] Processing: 004_natwest_group_plc_annual_report_2025.pdf
Completed: 436 pages extracted

[5/20] Processing: 005_standard_chartered_plc_annual_report_2025.pdf
Completed: 481 pages extracted

[6/20] Processing: 006_aviva_plc_annual_report_2025.pdf
Completed: 340 pages extracted

[7/20] Processing: 007_legal_and_general_group_plc_annual_report_2025.pdf
Completed: 259 pages extracted

[8/20] Processing: 008_m_and_g_plc_annual_report_2025.pdf
Completed: 345 pages extracted

[9/20] Processing: 009_admiral_group_plc_annual_report_2025.pdf
Completed: 336 pages extracted

[10/20] Processing: 010_aj_bell_plc_annual_report_2025.pdf
Completed: 168 pages extracted

[11/20] Proc

,file_name,page_count,pages_with_text,empty_or_image_only_pages,character_count,output_file,extraction_status,error_message
0,001_barclays_plc_annual_report_2025.pdf,494,494,0,3374218,001_barclays_plc_annual_report_2025.txt,Success,
1,002_hsbc_holdings_plc_annual_report_2025.pdf,372,370,2,2960156,002_hsbc_holdings_plc_annual_report_2025.txt,Success,
2,003_lloyds_banking_group_plc_annual_report_202...,327,327,0,2216103,003_lloyds_banking_group_plc_annual_report_202...,Success,
3,004_natwest_group_plc_annual_report_2025.pdf,436,435,1,3054471,004_natwest_group_plc_annual_report_2025.txt,Success,
4,005_standard_chartered_plc_annual_report_2025.pdf,481,481,0,2929081,005_standard_chartered_plc_annual_report_2025.txt,Success,
5,006_aviva_plc_annual_report_2025.pdf,340,340,0,3030552,006_aviva_plc_annual_report_2025.txt,Success,
6,007_legal_and_general_group_plc_annual_report_...,259,259,0,1714464,007_legal_and_general_group_plc_annual_report_...,Success,
7,008_m_and_g_plc_annual_report_2025.pdf,345,345,0,1776418,008_m_and_g_plc_annual_report_2025.txt,Success,
8,009_admiral_group_plc_annual_report_2025.pdf,336,336,0,1527607,009_admiral_group_plc_annual_report_2025.txt,Success,
9,010_aj_bell_plc_annual_report_2025.pdf,168,168,0,1119921,010_aj_bell_plc_annual_report_2025.txt,Success,
